In [56]:
import chromadb
import ollama
from groq import Groq
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import os

In [57]:
load_dotenv()
DIRECTORY_PATH = os.getenv("DIRECTORY_PATH")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
COLLECTION_NAME = os.getenv("COLLECTION_NAME")
DATABASE_PATH = os.getenv("DATABASE_PATH")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LOCAL_MODEL_NAME = os.getenv("LOCAL_MODEL_NAME")

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
client = chromadb.PersistentClient(path=DATABASE_PATH)
collection = client.get_or_create_collection(name=COLLECTION_NAME)
groq_client = Groq(
    # This is the default and can be omitted
    api_key=GROQ_API_KEY,
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6330.42it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#Yayy we're not at the Retrieval step (the R in RAG)
def retrieval(user_query):
    #Convert the question into an embedding using the exact same model we used earlier
    #model.encode returns a numpy array, but Chroma expects a list, so we use .tolist()
    query_embedding = embedding_model.encode(user_query).tolist()

    #Query the ChromaDB collection
    n_result=10
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_result,
    )

    #Display the results clearly
    #Chroma returns data in lists of lists to handle multiple queries at once, 
    #so we access the first element [0] of the documents, metadatas, and distances
    return results['documents'][0]

In [60]:
#THE G IN RAG. HOORAY
def generation(user_query):
    context_text = "\n\n---\n\n".join(retrieval(user_query))
    prompt = f"""
    You are a helpful and accurate assistant. Answer the user's question based ONLY on the provided context below. 
    If the context does not contain the answer, politely say "I don't have enough information in my knowledge base to answer that." 
    Do not use outside knowledge.

    Context:
    {context_text}

    """
    chat_completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": prompt
            },
            {
                "role": "user",
                "content": user_query,
            }
        ],
        model="llama-3.3-70b-versatile",
    )
    return chat_completion.choices[0].message.content
def generation_local(user_query):
    context_text = "\n\n---\n\n".join(retrieval(user_query))
    
    prompt = f"""
    You are a helpful and accurate assistant. Answer the user's question based ONLY on the provided context below. 
    If the context does not contain the answer, politely say "I don't have enough information in my knowledge base to answer that." 
    Do not use outside knowledge.

    Context:
    {context_text}
    """
    
    # Call your local Ollama server instead of Groq
    response = ollama.chat(
        model='llama3.2:3b', # Make sure this matches the model you downloaded
        messages=[
            {
                "role": "system",
                "content": prompt
            },
            {
                "role": "user",
                "content": user_query,
            }
        ]
    )
    
    # Ollama's response structure is slightly different from Groq's
    return response['message']['content']

# %%

In [61]:
print(generation('What is a Markov Decision Process?'))

A Markov Decision Process (MDP) is a mathematical framework used to model decision-making problems in situations where outcomes are partially random and partially under the control of the decision-maker. It is a tuple of four components: 

1. **States (𝒮)**: A finite set of states that the process can be in.
2. **Actions (𝒜)**: A finite set of actions that can be taken in each state.
3. **Rewards (ℛ)**: A set of rewards that can be received after taking an action in a state.
4. **Transition function (𝑝)**: A function that defines the dynamics of the MDP, specifying the probability of transitioning from one state to another and receiving a reward, given the current state and action.

In an MDP, the decision-maker (or agent) takes actions in a sequence of states, and the process generates rewards and next states based on the current state and action. The goal is to find a policy that maps states to actions in a way that maximizes the expected cumulative reward over time.
